In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit

# Read data in
df_11 = pd.read_csv(r"C:\Users\myles\.vscode\DAQ\SenHon\1_1_extended_flowcurve.csv")
df_10 = pd.read_csv(r"C:\Users\myles\.vscode\DAQ\SenHon\1_0_flowcurve.csv")
df_00 = pd.read_csv(r"C:\Users\myles\.vscode\DAQ\SenHon\0_0_flowcurve.csv")


# Model to fit yeild stress as a function of packign fraction
def yield_stress_model(phi, sigma_a, kappa, phi_alp, phi_mu):
    return sigma_a * np.log((phi_mu - phi_alp) / (phi - phi_alp)) ** (-1 / kappa)


def fit_yield_stress(phi_data, sigma_y_data, sigma_err=None):
    phi = np.asarray(phi_data, float)
    y   = np.asarray(sigma_y_data, float)

    mask = np.isfinite(phi) & np.isfinite(y)
    phi, y = phi[mask], y[mask]

    phi_min, phi_max = float(phi.min()), float(phi.max())

    lb = [0.0,  0.0,  0.0,           phi_max + 1e-4]
    ub = [1e6, 10.0,  phi_min - 1e-4, 0.80]

    if not (lb[2] < ub[2]):
        raise ValueError(f"Invalid phi_alp bounds: lb={lb[2]} ub={ub[2]} (phi_min={phi_min})")
    if not (lb[3] < ub[3]):
        raise ValueError(f"Invalid phi_mu bounds: lb={lb[3]} ub={ub[3]} (phi_max={phi_max})")

    bounds = (lb, ub)

    p0 = [
        float(np.median(y)),
        1.0,
        (lb[2] + ub[2]) / 2.0,
        (lb[3] + ub[3]) / 2.0
    ]

    sigma = None
    abs_sigma = False
    if sigma_err is not None:
        sigma = np.asarray(sigma_err, float)[mask]
        if np.any(~np.isfinite(sigma)) or np.any(sigma <= 0):
            raise ValueError("sigma_err must be finite and > 0 everywhere.")
        abs_sigma = True

    popt, pcov = curve_fit(
        yield_stress_model,
        phi, y,
        p0=p0,
        bounds=bounds,
        sigma=sigma,
        absolute_sigma=abs_sigma,
        maxfev=20000
    )
    perr = np.sqrt(np.diag(pcov))
    return popt, pcov, perr

# fit each regime
popt00, pcov00, perr00 = fit_yield_stress(df_00["phi"], df_00["sigma_y"])
popt10, pcov10, perr10 = fit_yield_stress(df_10["phi"], df_10["sigma_y"])
popt11, pcov11, perr11 = fit_yield_stress(df_11["phi"], df_11["sigma_y"])

print("0_0 popt:", popt00, "perr:", perr00)
print("1_0 popt:", popt10, "perr:", perr10)
print("1_1 popt:", popt11, "perr:", perr11)


#respective jamming fractions (right endpoints for the plotted curves)
phiJ_11 = 0.469
phiJ_10 = 0.577
phiJ_00 = 0.657

# Chosen to match the jamming plot for easy comparison
PHI_LEFT = 0.20



def safe_curve(phi_grid, popt):
    """
    Evaluate yield_stress_model only where it's defined:
      phi > phi_alp and phi < phi_mu
    Else return NaN (so matplotlib will break the line).
    """
    sigma_a, kappa, phi_alp, phi_mu = popt
    phi_grid = np.asarray(phi_grid, float)
    y = np.full_like(phi_grid, np.nan, dtype=float)
    ok = (phi_grid > phi_alp) & (phi_grid < phi_mu)
    if np.any(ok):
        y[ok] = yield_stress_model(phi_grid[ok], *popt)
    return y

def make_phi_grid(popt, phiJ_right, n=400):
    """
    Build a grid from PHI_LEFT to the jamming fraction, but
    also respect the model’s domain using a tiny buffer.
    """
    _, _, phi_alp, phi_mu = popt

    # start: cannot go <= phi_alp
    start = max(PHI_LEFT, phi_alp + 1e-6)

    # end: cannot go >= phi_mu
    end = min(phiJ_right, phi_mu - 1e-6)

    if end <= start:
        # nothing to plot (model domain doesn't overlap desired range)
        return np.array([np.nan, np.nan])

    return np.linspace(start, end, n)

# grids that *try* to go to phi=0.36 and to each phiJ, but remain valid
x_fit_00 = make_phi_grid(popt00, phiJ_00)
y_fit_00 = safe_curve(x_fit_00, popt00)

x_fit_10 = make_phi_grid(popt10, phiJ_10)
y_fit_10 = safe_curve(x_fit_10, popt10)

x_fit_11 = make_phi_grid(popt11, phiJ_11)
y_fit_11 = safe_curve(x_fit_11, popt11)



plt.figure(figsize=(8,6))

# raw data
plt.plot(df_11["phi"], df_11["sigma_y"], marker='o', color='red', linestyle='', label='data 1_1')
plt.plot(df_10["phi"], df_10["sigma_y"], marker='o', color='blue', linestyle='', label='data 1_0')
plt.plot(df_00["phi"], df_00["sigma_y"], marker='o', color='black', linestyle='', label='data 0_0')

# extrapolated fits (now extended)
plt.plot(x_fit_11, y_fit_11, linestyle='-', color='red', label='fit 1_1')
plt.plot(x_fit_10, y_fit_10, linestyle='-', color='blue', label='fit 1_0')
plt.plot(x_fit_00, y_fit_00, linestyle='-', color='black', label='fit 0_0')

# jamming fractions
plt.axvline(x=phiJ_11, linestyle='--', color='red', label=f'φ_J(1_1) = {phiJ_11:.4f}')
plt.axvline(x=phiJ_10, linestyle='--', color='blue', label=f'φ_J(1_0) = {phiJ_10:.4f}')
plt.axvline(x=phiJ_00, linestyle='--', color='black', label=f'φ_J(0_0) = {phiJ_00:.4f}')


plt.minorticks_on()
plt.tick_params(axis='both', which='both', direction='in', top=True, right=True)
plt.xlim(0.20, 0.70)
plt.yscale("log")
plt.ylim(8e-2, 1e2)



plt.xlabel("Packing fraction $\phi$", fontsize=14, fontweight='bold')
plt.ylabel("Yield stress $\hat{\sigma_y}$", fontsize=14, fontweight='bold')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.show()